# MarimbaVision
### CS 131 Final Project — Haley Lepe

Pipeline:
1. Extract audio from overhead video
2. Detect strike onsets from audio transients (librosa)
3. For each onset, grab the overhead frame at that timestamp
4. Identify note by color under mallet (naturals) or X position (sharps)
5. Output timestamped note sequence

## 1. Imports & Configuration

In [ ]:
import cv2
import numpy as np
import json
import os
import librosa
import subprocess
from dataclasses import dataclass
from typing import Optional

# ── Video path ────────────────────────────────────────────────────────────────
OVERHEAD_VIDEO = 'c_top_clap.MOV'   # update to your filename

# ── Mallet HSV range (purple) ─────────────────────────────────────────────────
MALLET_HSV_LOWER = np.array([125, 50,  100])
MALLET_HSV_UPPER = np.array([155, 255, 255])
MALLET_MIN_AREA  = 500
MALLET_MAX_AREA  = 8000

# ── Audio onset detection ─────────────────────────────────────────────────────
ONSET_MIN_GAP    = 0.3   # seconds — minimum time between strikes
ONSET_DELTA      = 0.3  # sensitivity — lower = more onsets detected

# ── Key layout ────────────────────────────────────────────────────────────────
SHARP_CHROMATIC_POSITIONS = [1, 3, 6, 8, 11, 13, 15, 18, 20, 23]

# ── Natural key color map (HSV) ───────────────────────────────────────────────
NATURAL_KEY_COLORS = {
    'G3':  (np.array([95, 200, 100]), np.array([110, 255, 200])),  # light blue
    'A3':  (np.array([0,   0,   180]), np.array([180, 30,  255])),  # white
    'B3':  (np.array([135, 80,  80]),  np.array([160, 255, 255])),  # magenta
    'C4':  (np.array([0,   150, 100]), np.array([8,   255, 255])),  # red
    'D4':  (np.array([8,   150, 100]), np.array([18,  255, 255])),  # orange
    'E4':  (np.array([22,  150, 100]), np.array([32,  255, 255])),  # yellow
    'F4':  (np.array([50,  100, 100]), np.array([80,  255, 255])),  # green
    'G4':  (np.array([95, 200, 100]), np.array([110, 255, 200])),  # light blue
    'A4':  (np.array([0,   0,   180]), np.array([180, 30,  255])),  # white
    'B4':  (np.array([135, 80,  80]),  np.array([160, 255, 255])),  # magenta
    'C5':  (np.array([0,   150, 100]), np.array([8,   255, 255])),  # red
    'D5':  (np.array([8,   150, 100]), np.array([18,  255, 255])),  # orange
    'E5':  (np.array([22,  150, 100]), np.array([32,  255, 255])),  # yellow
    'F5':  (np.array([50,  100, 100]), np.array([80,  255, 255])),  # green
    'G5':  (np.array([95, 200, 100]), np.array([110, 255, 200])),  # light blue
}

SHARP_ID_TO_NOTE = {
    5:'G#3', 7:'A#3', 10:'C#4', 12:'D#4', 15:'F#4',
    17:'G#4', 19:'A#4', 22:'C#5', 24:'D#5', 27:'F#5'
}

print('Config loaded')

✅ Config loaded


## 2. Data Structures

In [2]:
@dataclass
class Key:
    index: int
    note: str
    bbox: tuple
    center_x: float
    last_struck: int = -999

@dataclass
class Strike:
    frame: int
    timestamp: float
    note: str
    key_index: int
    mallet_pos: tuple
    identified_by: str = 'color'

_CHROMATIC = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

def build_note_labels(num_keys, start_midi=55):
    return [f"{_CHROMATIC[(start_midi+i)%12]}{(start_midi+i)//12-1}"
            for i in range(num_keys)]

print('✅ Data structures loaded')

✅ Data structures loaded


## 3. Audio Onset Detection

In [3]:
def extract_audio(video_path, audio_path='temp_audio.wav'):
    """Extract audio track from video using ffmpeg."""
    cmd = ['ffmpeg', '-y', '-i', video_path,
           '-vn', '-acodec', 'pcm_s16le',
           '-ar', '44100', '-ac', '1', audio_path]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f'ffmpeg failed: {result.stderr.decode()}')
    print(f'[AUDIO] Extracted to {audio_path}')
    return audio_path


def detect_strike_times(video_path, min_gap=ONSET_MIN_GAP, delta=ONSET_DELTA):
    """
    Detect strike timestamps from audio using librosa onset detection.
    Returns list of timestamps in seconds.
    """
    audio_path = extract_audio(video_path)
    y, sr = librosa.load(audio_path, sr=None, mono=True)

    # Onset strength envelope
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)

    # Detect onsets — delta controls sensitivity
    onset_frames = librosa.onset.onset_detect(
        y=y, sr=sr,
        onset_envelope=onset_env,
        delta=delta,
        units='frames'
    )
    onset_times = librosa.frames_to_time(onset_frames, sr=sr)

    # Filter: remove onsets too close together (same strike detected twice)
    filtered = []
    last_t = -999
    for t in onset_times:
        if t - last_t >= min_gap:
            filtered.append(float(t))
            last_t = t

    print(f'[AUDIO] Raw onsets: {len(onset_times)} → filtered: {len(filtered)}')
    for i, t in enumerate(filtered):
        print(f'  onset {i+1}: {t:.3f}s')

    # Clean up temp file
    os.remove(audio_path)
    return filtered

print('✅ Audio onset detection loaded')

✅ Audio onset detection loaded


## 4. Key Detection (ArUco)

In [4]:
def _interpolate_keys_chromatic(start, end, target_notes, total_semitones,
                                 key_w, key_h, y_offset=0,
                                 x_offset_start=0, x_offset_end=0):
    keys = []
    for i, note in enumerate(target_notes):
        if '#' in note:
            idx = SHARP_CHROMATIC_POSITIONS[i]
            t   = idx / max(total_semitones - 1, 1)
        else:
            t = i / max(len(target_notes) - 1, 1)
        x_off = x_offset_start + t * (x_offset_end - x_offset_start)
        cx    = start[0] + t * (end[0] - start[0]) + x_off
        cy    = start[1] + t * (end[1] - start[1]) + y_offset
        bbox  = (int(cx - key_w//2), int(cy - key_h//2), key_w, key_h)
        keys.append(Key(len(keys), note, bbox, float(cx)))
    return keys


def detect_keys_from_aruco(frame, debug=False):
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    params     = cv2.aruco.DetectorParameters()
    params.minMarkerPerimeterRate      = 0.02
    params.polygonalApproxAccuracyRate = 0.08
    detector   = cv2.aruco.ArucoDetector(aruco_dict, params)
    corners, ids, _ = detector.detectMarkers(frame)

    if ids is None or len(ids) < 2:
        print('[WARN] Not enough ArUco markers')
        return []
    marker_centers = {int(ids[i][0]): corners[i][0].mean(axis=0)
                  for i in range(len(ids))
                  if int(ids[i][0]) <= 3}
    print(f'[INFO] ArUco markers: {list(marker_centers.keys())}')

    all_notes     = build_note_labels(25, start_midi=55)
    sharp_notes   = [n for n in all_notes if '#' in n]
    natural_notes = [n for n in all_notes if '#' not in n]
    all_keys      = []

    if 0 in marker_centers and 1 in marker_centers:
        all_keys.extend(_interpolate_keys_chromatic(
            start=marker_centers[0], end=marker_centers[1],
            target_notes=sharp_notes, total_semitones=25,
            key_w=60, key_h=120, y_offset=120,
            x_offset_start=30, x_offset_end=-30))

    if 2 in marker_centers and 3 in marker_centers:
        all_keys.extend(_interpolate_keys_chromatic(
            start=marker_centers[2], end=marker_centers[3],
            target_notes=natural_notes, total_semitones=25,
            key_w=60, key_h=120, y_offset=-120))

    for i, k in enumerate(all_keys):
        k.index = i

    if debug:
        vis = frame.copy()
        for mid, pt in marker_centers.items():
            cv2.circle(vis, (int(pt[0]), int(pt[1])), 10, (0,0,255), -1)
        for k in all_keys:
            x, y, w, h = k.bbox
            color = (255,100,0) if '#' in k.note else (0,255,0)
            cv2.rectangle(vis, (x,y), (x+w,y+h), color, 2)
            cv2.putText(vis, k.note, (x, y-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.3, color, 1)
        cv2.imwrite('debug_keys.jpg', vis)
        print(f'[DEBUG] {len(all_keys)} keys → debug_keys.jpg')

    return all_keys


def remap_sharp_keys_from_naturals(keys):
    naturals = {k.note: k for k in keys if '#' not in k.note}
    
    sharp_neighbors = {
        'G#3': ('G3', 'A3'),
        'A#3': ('A3', 'B3'),
        'C#4': ('C4', 'D4'),
        'D#4': ('D4', 'E4'),
        'F#4': ('F4', 'G4'),
        'G#4': ('G4', 'A4'),
        'A#4': ('A4', 'B4'),
        'C#5': ('C5', 'D5'),
        'D#5': ('D5', 'E5'),
        'F#5': ('F5', 'G5'),
    }
    
    # Compute how far above naturals the sharps should sit
    # Sharp keys are physically higher up (smaller Y value) than naturals
    natural_ys = [k.bbox[1] + k.bbox[3]/2 for k in keys if '#' not in k.note]
    sharp_ys   = [k.bbox[1] + k.bbox[3]/2 for k in keys if '#' in k.note]
    
    if natural_ys and sharp_ys:
        # Use the actual difference detected from ArUco interpolation as baseline
        y_shift = np.mean(sharp_ys) - np.mean(natural_ys)
    else:
        y_shift = -150  # fallback: sharps sit ~150px above naturals
    
    for k in keys:
        if '#' not in k.note:
            continue
        if k.note not in sharp_neighbors:
            continue
        left_note, right_note = sharp_neighbors[k.note]
        if left_note in naturals and right_note in naturals:
            # X = midpoint of neighboring naturals
            new_cx = (naturals[left_note].center_x + 
                      naturals[right_note].center_x) / 2
            # Y = natural key Y + the sharp row offset
            nat_cy = (naturals[left_note].bbox[1] + naturals[left_note].bbox[3]/2 +
                      naturals[right_note].bbox[1] + naturals[right_note].bbox[3]/2) / 2
            new_cy = nat_cy + y_shift
            
            w, h   = k.bbox[2], k.bbox[3]
            k.center_x = new_cx
            k.bbox = (int(new_cx - w/2), int(new_cy - h/2), w, h)
    
    return keys

def detect_sharp_key_positions(frame, num_sharps=10):
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    blue_lower = np.array([100, 80, 60])
    blue_upper = np.array([130, 255, 200])
    
    mask   = cv2.inRange(hsv, blue_lower, blue_upper)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    # Only look in top portion where sharp keys are
    fh, fw = frame.shape[:2]
    mask[int(fh*0.50):, :] = 0
    
    # Use column-wise projection to find gaps between keys
    # Each column's sum — low sum = gap between keys
    col_sums = mask.sum(axis=0).astype(float)
    
    # Smooth the projection
    col_sums = np.convolve(col_sums, np.ones(5)/5, mode='same')
    
    # Find peaks (key centers) in the projection
    from scipy.signal import find_peaks
    peaks, properties = find_peaks(
        col_sums,
        distance=fw//15,      # minimum distance between keys
        height=col_sums.max() * 0.3  # minimum height to count
    )
    
    print(f'[SHARP] Column projection found {len(peaks)} peaks')
    
    if len(peaks) == 0:
        return []
    
    # Build key regions from peaks
    key_centers = []
    sharp_y_top    = int(fh * 0.05)
    sharp_y_bottom = int(fh * 0.45)
    key_h = sharp_y_bottom - sharp_y_top
    key_w = fw // 18  # approximate key width
    
    for px in peaks:
        cx = int(px)
        cy = (sharp_y_top + sharp_y_bottom) // 2
        x  = max(0, cx - key_w//2)
        y  = sharp_y_top
        key_centers.append((cx, cy, x, y, key_w, key_h))
    
    return key_centers


def remap_sharps_from_blue_detection(keys, frame):
    """
    Replace sharp key X positions using direct blue color detection.
    """
    sharp_notes = ['G#3','A#3','C#4','D#4','F#4','G#4','A#4','C#5','D#5','F#5']
    sharp_keys  = [k for k in keys if '#' in k.note]
    
    detected = detect_sharp_key_positions(frame)
    
    if len(detected) == 0:
        print('[WARN] No blue keys detected — keeping ArUco positions')
        return keys
    
    print(f'[SHARP] Mapping {len(detected)} detected regions to {len(sharp_keys)} sharp keys')
    
    # If we got exactly 10, map directly
    if len(detected) == len(sharp_keys):
        for k, (cx, cy, x, y, w, h) in zip(sharp_keys, detected):
            k.center_x = float(cx)
            k.bbox     = (x, y, w, h)
        print('[SHARP] Direct 1:1 mapping applied')
    
    # If we got fewer, map what we have
    elif len(detected) < len(sharp_keys):
        print(f'[SHARP] Only {len(detected)} detected — partial mapping')
        for i, (cx, cy, x, y, w, h) in enumerate(detected):
            if i < len(sharp_keys):
                sharp_keys[i].center_x = float(cx)
                sharp_keys[i].bbox     = (x, y, w, h)
    
    return keys

print("key detection done")


key detection done


## 5. Mallet Tracking

In [5]:
def track_mallet(frame):
    hsv    = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask   = cv2.inRange(hsv, MALLET_HSV_LOWER, MALLET_HSV_UPPER)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    best, best_area = None, 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if not (MALLET_MIN_AREA < area < MALLET_MAX_AREA):
            continue
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue
        if 4*np.pi*area/(perimeter**2) < 0.6:
            continue
        if area > best_area:
            best_area, best = area, cnt
    if best is None:
        return None
    # After finding best contour, before returning:
    (cx, cy), r = cv2.minEnclosingCircle(best)

    # Reject if in top 25% of frame — that's the sharp key area not playing area
    fh = frame.shape[0]
    if cy < fh * 0.25:
        return None

    return int(cx), int(cy), int(r)

print('✅ Mallet tracking loaded')

✅ Mallet tracking loaded


## 6. Color-Based Note Identification

In [6]:
def identify_note_by_color(frame, mallet_cx, mallet_cy, mallet_r, natural_keys):
    """Sample color under mallet, match to natural key. Returns note or None."""
    hsv      = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    sample_r = max(5, mallet_r // 3)
    x1 = max(0, mallet_cx - sample_r)
    x2 = min(frame.shape[1], mallet_cx + sample_r)
    y1 = min(frame.shape[0], mallet_cy + mallet_r)           # start at bottom of ball
    y2 = min(frame.shape[0], mallet_cy + mallet_r + sample_r*2)  # extend below
    roi = hsv[y1:y2, x1:x2]
    if roi.size == 0:
        return None, None

    best_note, best_score = None, 0
    for note, (lower, upper) in NATURAL_KEY_COLORS.items():
        if note in ('C4','C5'):
            m1 = cv2.inRange(roi, np.array([0,150,100]),   np.array([8,255,255]))
            m2 = cv2.inRange(roi, np.array([172,150,100]), np.array([180,255,255]))
            mask = cv2.bitwise_or(m1, m2)
        else:
            mask = cv2.inRange(roi, lower, upper)
        score = cv2.countNonZero(mask)
        if score > best_score:
            best_score, best_note = score, note

    total = (2*sample_r)**2
    if best_score < total * 0.30:
        return None, None

    # Disambiguate same-color notes by X position
    if best_note and natural_keys:
        base       = best_note[:-1]
        candidates = [k for k in natural_keys if k.note[:-1] == base]
        if candidates:
            best_key = min(candidates, key=lambda k: abs(k.center_x - mallet_cx))
            return best_key.note, 'color'

    return best_note, 'color'


def identify_note(frame, mallet_cx, mallet_cy, mallet_r, keys):
    naturals = [k for k in keys if '#' not in k.note]

    # 1. Color for naturals
    note, method = identify_note_by_color(
        frame, mallet_cx, mallet_cy, mallet_r, naturals)
    if note:
        return note, method

    # 2. ArUco for sharps — pass mallet X for disambiguation
    note, method = identify_sharp_by_aruco(frame, mallet_cx)
    if note:
        return note, method

    # 3. Position fallback
    best = min(keys, key=lambda k: abs(k.center_x - mallet_cx), default=None)
    return (best.note, 'position') if best else (None, None)


# def identify_sharp_by_position(frame, keys):
#     """
#     Track mallet without Y bounds restriction to find sharp key position.
#     """
#     hsv    = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
#     mask   = cv2.inRange(hsv, MALLET_HSV_LOWER, MALLET_HSV_UPPER)
#     kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
#     mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
#     mask   = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)
#     contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
#                                     cv2.CHAIN_APPROX_SIMPLE)
#     best, best_area = None, 0
#     for cnt in contours:
#         area = cv2.contourArea(cnt)
#         if not (MALLET_MIN_AREA < area < MALLET_MAX_AREA):
#             continue
#         perimeter = cv2.arcLength(cnt, True)
#         if perimeter == 0:
#             continue
#         if 4*np.pi*area/(perimeter**2) < 0.6:
#             continue
#         if area > best_area:
#             best_area, best = area, cnt

#     if best is None:
#         return None, None

#     (cx, cy), r = cv2.minEnclosingCircle(best)
#     cx, cy = int(cx), int(cy)

#     # Only accept if in sharp key row (upper portion of frame)
#     fh = frame.shape[0]
#     if cy > fh * 0.25:
#         return None, None  # not in sharp area

#     # Find nearest sharp key by X
#     sharps = [k for k in keys if '#' in k.note]
#     if not sharps:
#         return None, None

#     best_sharp = min(sharps, key=lambda k: abs(k.center_x - cx))
#     return best_sharp.note, 'aruco_x'

def identify_sharp_by_aruco(frame, mallet_cx):
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    params     = cv2.aruco.DetectorParameters()
    params.minMarkerPerimeterRate = 0.01
    detector   = cv2.aruco.ArucoDetector(aruco_dict, params)
    corners, ids, _ = detector.detectMarkers(frame)
    
    if ids is None:
        return None, None
    
    # Build dict of detected sharp markers and their X positions
    sharp_markers = {}
    for i, corner in enumerate(corners):
        mid = int(ids[i][0])
        if mid in SHARP_ID_TO_NOTE:
            sharp_markers[mid] = float(corner[0].mean(axis=0)[0])
    
    if not sharp_markers:
        return None, None
    
    # Pick closest to mallet X
    best_id   = min(sharp_markers, key=lambda k: abs(sharp_markers[k] - mallet_cx))
    best_dist = abs(sharp_markers[best_id] - mallet_cx)
    
    print(f'  [SHARP] Detected: {[SHARP_ID_TO_NOTE[k] for k in sharp_markers]}')
    print(f'  [SHARP] Mallet X={mallet_cx}, closest={SHARP_ID_TO_NOTE[best_id]} (dist={best_dist:.0f}px)')
    
    if best_dist > 200:
        return None, None
    
    return SHARP_ID_TO_NOTE[best_id], 'aruco_sharp'

print('✅ Note identification loaded')

✅ Note identification loaded


## 7. Vanishing Points & Camera Matrix K

In [7]:
def line_intersection(p1, p2, p3, p4):
    a0h = np.append(np.array(p1,dtype=float),1.)
    a1h = np.append(np.array(p2,dtype=float),1.)
    b0h = np.append(np.array(p3,dtype=float),1.)
    b1h = np.append(np.array(p4,dtype=float),1.)
    p   = np.cross(np.cross(a0h,a1h), np.cross(b0h,b1h))
    if abs(p[2]) < 1e-10:
        return None
    return (p[:2]/p[2]).astype(int)

def compute_camera_matrix(frame, marker_centers):
    fh, fw = frame.shape[:2]
    oc     = np.array([fw/2, fh/2])
    if not all(i in marker_centers for i in range(4)):
        return None, None, None
    vp1 = line_intersection(marker_centers[0], marker_centers[1],
                             marker_centers[2], marker_centers[3])
    vp2 = line_intersection(marker_centers[0], marker_centers[2],
                             marker_centers[1], marker_centers[3])
    if vp1 is None or vp2 is None:
        return None, None, None
    cx, cy = oc
    val = -((vp1[0]-cx)*(vp2[0]-cx)+(vp1[1]-cy)*(vp2[1]-cy))
    if val < 0:
        print('[WARN] VPs not orthogonal')
        return None, None, None
    f = float(np.sqrt(val))
    K = np.array([[f,0,cx],[0,f,cy],[0,0,1]])
    print(f'[INFO] VP1={vp1} VP2={vp2}')
    print(f'[INFO] Focal length: {f:.1f}px')
    print(f'[INFO] K:\n{K}')
    return K, vp1, vp2

def find_vanishing_point(frame, marker_centers, debug=False):
    vp = line_intersection(marker_centers[0], marker_centers[2],
                            marker_centers[1], marker_centers[3])
    if debug and vp is not None:
        fh, fw     = frame.shape[:2]
        pad_left   = max(0,-vp[0])+150
        pad_right  = max(0,vp[0]-fw)+150
        pad_top    = max(0,-vp[1])+150
        pad_bottom = max(0,vp[1]-fh)+150
        canvas = cv2.copyMakeBorder(frame,pad_top,pad_bottom,pad_left,pad_right,
                                    cv2.BORDER_CONSTANT,value=(15,15,15))
        def sh(pt): return (int(pt[0])+pad_left, int(pt[1])+pad_top)
        vp_s = sh(vp)
        for pt in marker_centers.values():
            cv2.line(canvas, sh(pt), vp_s, (0,200,255), 3)
        cv2.circle(canvas, vp_s, 8, (0,0,255), -1)
        cv2.putText(canvas, f'VP ({vp[0]},{vp[1]})',
                    (vp_s[0]+15,vp_s[1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
        cv2.imwrite('vanishing_point.jpg', canvas)
        print(f'[INFO] Vanishing point → vanishing_point.jpg')
    return vp

print('✅ Camera matrix loaded')

✅ Camera matrix loaded


## 8. Main Pipeline (Audio-Driven)

In [8]:
def process_video(video_path, output_path='notes.json', debug=False):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f'Cannot open: {video_path}')

    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'[INFO] {os.path.basename(video_path)} | {fps:.1f}fps | {total/fps:.1f}s')

    # ── Step 1: Detect strike times from audio ────────────────────────────────
    print('\n[STEP 1] Audio onset detection...')
    strike_times = detect_strike_times(video_path)
    print(f'[AUDIO] Found {len(strike_times)} strikes')

    # ── Step 2: Key detection — skip first 1.5s ───────────────────────────────
    print('\n[STEP 2] Key detection...')
    keys = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fps * 1.5))
    for _ in range(60):
        ret, frame = cap.read()
        if not ret:
            break
        keys = detect_keys_from_aruco(frame)
        keys = remap_sharp_keys_from_naturals(keys)
        if len(keys) >= 20:
            break
    print(f'[INFO] {len(keys)} keys mapped')

    if not keys:
        print('[ERROR] No keys detected — check markers')
        cap.release()
        return []

    # ── Step 3: For each onset, grab frame and identify note ──────────────────
    print('\n[STEP 3] Identifying notes at each onset...')
    all_strikes = []
    debug_frames = []

    for i, t in enumerate(strike_times):
        frame_no = int(t * fps)

        # Scan forward from onset to find lowest mallet position (actual contact)
        best_frame = None
        best_cy    = -1

        for offset in range(0, 12):  # look up to 8 frames ahead
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no + offset)
            ret, f = cap.read()
            if not ret:
                break
            m = track_mallet(f)
            if m:
                _, cy, _ = m
                if cy > best_cy:  # higher Y = lower in frame = closer to key
                    best_cy    = cy
                    best_frame = f.copy()

        if best_frame is None:
            print(f'  Strike {i+1} @ {t:.3f}s — mallet not visible')
            continue

        frame  = best_frame
        mallet = track_mallet(frame)
        if mallet is None:
            continue

        cx, cy, r = mallet
        note, method = identify_note(frame, cx, cy, r, keys)

        if note is None:
            print(f'  Strike {i+1} @ {t:.3f}s — note not identified')
            continue

        key = next((k for k in keys if k.note == note), None)
        if key is None:
            continue

        s = Strike(frame_no, t, note, key.index, (cx, cy), method)
        all_strikes.append(s)
        print(f'  🎵 Strike {i+1} @ {t:.3f}s → {note} [{method}]')

        if debug:
            vis = frame.copy()
            cv2.circle(vis, (cx,cy), r, (0,0,255), 2)
            cv2.circle(vis, (cx,cy), r//3, (255,255,0), 2)
            cv2.putText(vis, f'{note} [{method}]', (cx+10,cy),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
            for k in keys:
                x, y, w, h = k.bbox
                c = (0,255,255) if k.note==note else \
                    (255,100,0) if '#' in k.note else (0,255,0)
                cv2.rectangle(vis,(x,y),(x+w,y+h),c,2)
            debug_frames.append((t, vis))

    cap.release()

    if debug:
        for idx, (t, vis) in enumerate(debug_frames):
            cv2.imwrite(f'strike_{idx+1}_{t:.2f}s.jpg', vis)
        print(f'[DEBUG] Saved {len(debug_frames)} strike frames')

    # ── Output ─────────────────────────────────────────────────────────────────
    data = {
        'video': os.path.basename(video_path),
        'fps': fps,
        'strikes': [{
            'frame': s.frame,
            'timestamp': round(s.timestamp, 3),
            'note': s.note,
            'key_index': s.key_index,
            'mallet_pos': s.mallet_pos,
            'identified_by': s.identified_by
        } for s in all_strikes]
    }
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=2)

    print(f'\n[DONE] {len(all_strikes)} strikes → {output_path}')
    return all_strikes

## 9. Test Audio Detection First

In [9]:
# ── Run audio detection in isolation to verify onset count ───────────────────
# Tune ONSET_DELTA in Section 1 if you get too many or too few:
#   too many onsets → increase ONSET_DELTA (e.g. 0.1)
#   too few onsets  → decrease ONSET_DELTA (e.g. 0.04)

test_times = detect_strike_times(OVERHEAD_VIDEO)
print(f'\nExpected: 4 strikes')
print(f'Detected: {len(test_times)} strikes')
print(f'Timestamps: {[round(t,3) for t in test_times]}')

[AUDIO] Extracted to temp_audio.wav
[AUDIO] Raw onsets: 5 → filtered: 5
  onset 1: 0.139s
  onset 2: 3.680s
  onset 3: 4.470s
  onset 4: 5.213s
  onset 5: 5.898s

Expected: 4 strikes
Detected: 5 strikes
Timestamps: [0.139, 3.68, 4.47, 5.213, 5.898]


## 10. Run Full Pipeline

In [10]:
strikes = process_video(
    video_path  = OVERHEAD_VIDEO,
    output_path = 'notes.json',
    debug       = True   # saves strike_1_Xs.jpg etc for each hit
)

if strikes:
    print('\n=== NOTE SEQUENCE ===')
    for s in strikes:
        print(f'  {s.timestamp:.3f}s → {s.note} [{s.identified_by}]')
else:
    print('No strikes detected')

[INFO] c_top_clap.MOV | 58.5fps | 15.2s

[STEP 1] Audio onset detection...
[AUDIO] Extracted to temp_audio.wav
[AUDIO] Raw onsets: 5 → filtered: 5
  onset 1: 0.139s
  onset 2: 3.680s
  onset 3: 4.470s
  onset 4: 5.213s
  onset 5: 5.898s
[AUDIO] Found 5 strikes

[STEP 2] Key detection...
[INFO] ArUco markers: [1, 2, 3, 0]
[INFO] 25 keys mapped

[STEP 3] Identifying notes at each onset...
  Strike 1 @ 0.139s — mallet not visible
  🎵 Strike 2 @ 3.680s → C4 [color]
  🎵 Strike 3 @ 4.470s → E4 [color]
  🎵 Strike 4 @ 5.213s → G4 [color]
  🎵 Strike 5 @ 5.898s → C5 [color]
[DEBUG] Saved 4 strike frames

[DONE] 4 strikes → notes.json

=== NOTE SEQUENCE ===
  3.680s → C4 [color]
  4.470s → E4 [color]
  5.213s → G4 [color]
  5.898s → C5 [color]


In [11]:
import cv2
import numpy as np

cap = cv2.VideoCapture('b_chord_extra.mov')
fps = cap.get(cv2.CAP_PROP_FPS)

# Check frames around onset 2 @ 2.995s
onset_t = 2.995
frame_no = int(onset_t * fps)
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

for offset in range(-2, 15):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no + offset)
    ret, f = cap.read()
    if not ret:
        break
    
    # Track WITHOUT Y bounds
    hsv    = cv2.cvtColor(f, cv2.COLOR_BGR2HSV)
    mask   = cv2.inRange(hsv, MALLET_HSV_LOWER, MALLET_HSV_UPPER)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if MALLET_MIN_AREA < area < MALLET_MAX_AREA:
            (cx, cy), r = cv2.minEnclosingCircle(cnt)
            pct = cy / fh * 100
            rejected = '❌ REJECTED (Y bounds)' if cy < fh * 0.25 else '✅ accepted'
            print(f'  offset={offset:+d} | cy={int(cy)} ({pct:.0f}% down) | {rejected}')

cap.release()

  offset=-2 | cy=653 (60% down) | ✅ accepted
  offset=-1 | cy=659 (61% down) | ✅ accepted
  offset=+0 | cy=663 (61% down) | ✅ accepted
  offset=+1 | cy=664 (62% down) | ✅ accepted
  offset=+2 | cy=665 (62% down) | ✅ accepted
  offset=+3 | cy=662 (61% down) | ✅ accepted
  offset=+4 | cy=657 (61% down) | ✅ accepted
  offset=+5 | cy=652 (60% down) | ✅ accepted
  offset=+6 | cy=652 (60% down) | ✅ accepted
  offset=+7 | cy=655 (61% down) | ✅ accepted
  offset=+8 | cy=657 (61% down) | ✅ accepted
  offset=+9 | cy=661 (61% down) | ✅ accepted
  offset=+10 | cy=664 (62% down) | ✅ accepted
  offset=+11 | cy=671 (62% down) | ✅ accepted
  offset=+12 | cy=684 (63% down) | ✅ accepted
  offset=+13 | cy=711 (66% down) | ✅ accepted
  offset=+14 | cy=727 (67% down) | ✅ accepted


In [12]:
import inspect
print(inspect.getsource(track_mallet))

def track_mallet(frame):
    hsv    = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask   = cv2.inRange(hsv, MALLET_HSV_LOWER, MALLET_HSV_UPPER)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    best, best_area = None, 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if not (MALLET_MIN_AREA < area < MALLET_MAX_AREA):
            continue
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue
        if 4*np.pi*area/(perimeter**2) < 0.6:
            continue
        if area > best_area:
            best_area, best = area, cnt
    if best is None:
        return None
    # After finding best contour, before returning:
    (cx, cy), r = cv2.minEnclosingCircle(best)

    # Re

In [13]:
import inspect
src = inspect.getsource(identify_note_by_color)
# Find the line with best_score < total
for line in src.split('\n'):
    if 'best_score' in line and 'total' in line:
        print(line)

    if best_score < total * 0.30:


In [14]:
strikes = process_video(
    video_path  = 'b_chord_extra.mov',
    output_path = 'notesbex.json',
    debug       = True
)



[INFO] b_chord_extra.mov | 58.0fps | 8.0s

[STEP 1] Audio onset detection...
[AUDIO] Extracted to temp_audio.wav
[AUDIO] Raw onsets: 5 → filtered: 5
  onset 1: 0.139s
  onset 2: 2.357s
  onset 3: 3.216s
  onset 4: 3.936s
  onset 5: 4.586s
[AUDIO] Found 5 strikes

[STEP 2] Key detection...
[INFO] ArUco markers: [3, 2, 1, 0]
[INFO] 25 keys mapped

[STEP 3] Identifying notes at each onset...
  Strike 1 @ 0.139s — mallet not visible
  [SHARP] Detected: ['A#3', 'A#4', 'G#4', 'G#3']
  [SHARP] Mallet X=413, closest=A#3 (dist=18px)
  🎵 Strike 2 @ 2.357s → A#3 [aruco_sharp]
  🎵 Strike 3 @ 3.216s → C4 [color]
  🎵 Strike 4 @ 3.936s → F4 [color]
  [SHARP] Detected: ['A#4', 'G#4', 'A#3', 'G#3']
  [SHARP] Mallet X=1106, closest=A#4 (dist=16px)
  🎵 Strike 5 @ 4.586s → A#4 [aruco_sharp]
[DEBUG] Saved 4 strike frames

[DONE] 4 strikes → notesbex.json


In [15]:
# Sample G4 key color from a clean frame
frame = cv2.imread('background_still_purple.jpg')
hsv   = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

# Adjust these coordinates to point at the center of your G4 key
# Look at debug_keys.jpg to find approximate pixel location
g4_region = hsv[500:530, 800:830]  # adjust to hit G4
case_region = hsv[100:130, 100:130]  # adjust to hit the blue case

print(f'G4 key HSV mean:  {g4_region.mean(axis=(0,1)).astype(int)}')
print(f'Case HSV mean:    {case_region.mean(axis=(0,1)).astype(int)}')

G4 key HSV mean:  [101 247 153]
Case HSV mean:    [172  30  32]


In [16]:
cap = cv2.VideoCapture('b_chord_extra.mov')
fps = cap.get(cv2.CAP_PROP_FPS)

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
params     = cv2.aruco.DetectorParameters()
params.minMarkerPerimeterRate = 0.01
detector   = cv2.aruco.ArucoDetector(aruco_dict, params)

onset_times = [2.357, 3.216, 3.936, 4.586]

for t in onset_times:
    frame_no = int(t * fps)
    print(f'\nOnset @ {t}s:')
    for offset in range(-5, 5):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no + offset)
        ret, frame = cap.read()
        if not ret:
            break
        corners, ids, _ = detector.detectMarkers(frame)
        if ids is not None:
            sharp_ids = [i for i in ids.flatten() if i in SHARP_ID_TO_NOTE]
            if sharp_ids:
                print(f'  offset={offset:+d} → sharp markers detected: {[SHARP_ID_TO_NOTE[i] for i in sharp_ids]}')

cap.release()


Onset @ 2.357s:
  offset=-5 → sharp markers detected: ['A#4', 'G#4']
  offset=-4 → sharp markers detected: ['A#4', 'G#4']
  offset=-3 → sharp markers detected: ['A#4', 'A#3', 'G#4']
  offset=-2 → sharp markers detected: ['A#3', 'A#4', 'G#4']
  offset=-1 → sharp markers detected: ['A#3', 'A#4', 'G#4', 'G#3']
  offset=+0 → sharp markers detected: ['A#3', 'A#4', 'G#4', 'G#3']
  offset=+1 → sharp markers detected: ['A#3', 'A#4', 'G#4', 'G#3']
  offset=+2 → sharp markers detected: ['A#3', 'A#4', 'G#4', 'G#3']
  offset=+3 → sharp markers detected: ['A#4', 'G#4', 'G#3', 'A#3']
  offset=+4 → sharp markers detected: ['A#3', 'A#4', 'G#4', 'G#3']

Onset @ 3.216s:
  offset=-5 → sharp markers detected: ['A#3', 'A#4', 'G#4', 'G#3']
  offset=-4 → sharp markers detected: ['A#3', 'A#4', 'G#4', 'G#3']
  offset=-3 → sharp markers detected: ['A#3', 'A#4', 'G#4', 'G#3']
  offset=-2 → sharp markers detected: ['A#3', 'A#4', 'G#4', 'G#3']
  offset=-1 → sharp markers detected: ['A#3', 'A#4', 'G#4', 'G#3']
  o

## 11. Still Image Tests

In [17]:
# ── Key detection test ────────────────────────────────────────────────────────
frame = cv2.imread('background_still_purple.jpg')
keys = detect_keys_from_aruco(frame)
keys = remap_sharps_from_blue_detection(keys, frame)
print(f'{len(keys)} keys detected')

# ── Vanishing point & K ───────────────────────────────────────────────────────
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
detector   = cv2.aruco.ArucoDetector(aruco_dict, cv2.aruco.DetectorParameters())
corners, ids, _ = detector.detectMarkers(frame)
if ids is not None:
    mc = {int(ids[i][0]): corners[i][0].mean(axis=0) for i in range(len(ids))}
    K, vp1, vp2 = compute_camera_matrix(frame, mc)
    find_vanishing_point(frame, mc, debug=True)

[INFO] ArUco markers: [1, 2, 3, 0]
[SHARP] Column projection found 9 peaks
[SHARP] Mapping 9 detected regions to 10 sharp keys
[SHARP] Only 9 detected — partial mapping
25 keys detected
[WARN] VPs not orthogonal
[INFO] Vanishing point → vanishing_point.jpg


In [18]:
# ── Section 12: Sharp Key Calibration Check ───────────────────────────────────
# Run this to visually verify sharp key detection before running full pipeline
# Saves calibration_check.jpg — check that blue boxes align with actual blue keys

frame  = cv2.imread('background_still_purple.jpg')
keys   = detect_keys_from_aruco(frame)
keys   = remap_sharps_from_blue_detection(keys, frame)

# Draw result
vis = frame.copy()

# Show raw blue detection in red
detected = detect_sharp_key_positions(frame)
for cx, cy, x, y, w, h in detected:
    cv2.rectangle(vis, (x,y), (x+w,y+h), (0,0,255), 2)
    cv2.circle(vis, (cx,cy), 5, (0,0,255), -1)

# Show final key boxes
for k in keys:
    x, y, w, h = k.bbox
    color = (255,100,0) if '#' in k.note else (0,255,0)
    cv2.rectangle(vis, (x,y), (x+w,y+h), color, 2)
    cv2.putText(vis, k.note, (x, y-4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.3, color, 1)

cv2.imwrite('calibration_check.jpg', vis)
print('Saved calibration_check.jpg — check that boxes align with keys')
print(f'Sharp keys detected: {len(detected)}')
print(f'Total keys mapped: {len(keys)}')

[INFO] ArUco markers: [1, 2, 3, 0]
[SHARP] Column projection found 9 peaks
[SHARP] Mapping 9 detected regions to 10 sharp keys
[SHARP] Only 9 detected — partial mapping
[SHARP] Column projection found 9 peaks
Saved calibration_check.jpg — check that boxes align with keys
Sharp keys detected: 9
Total keys mapped: 25


In [19]:
import cv2
import numpy as np

cap = cv2.VideoCapture('b_chord_extra.mov')
fps = cap.get(cv2.CAP_PROP_FPS)

# Check frames around onset 2 @ 2.995s
onset_t = 2.995
frame_no = int(onset_t * fps)
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

for offset in range(-2, 15):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no + offset)
    ret, f = cap.read()
    if not ret:
        break
    
    # Track WITHOUT Y bounds
    hsv    = cv2.cvtColor(f, cv2.COLOR_BGR2HSV)
    mask   = cv2.inRange(hsv, MALLET_HSV_LOWER, MALLET_HSV_UPPER)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if MALLET_MIN_AREA < area < MALLET_MAX_AREA:
            (cx, cy), r = cv2.minEnclosingCircle(cnt)
            pct = cy / fh * 100
            rejected = '❌ REJECTED (Y bounds)' if cy < fh * 0.25 else '✅ accepted'
            print(f'  offset={offset:+d} | cy={int(cy)} ({pct:.0f}% down) | {rejected}')

cap.release()

  offset=-2 | cy=653 (60% down) | ✅ accepted
  offset=-1 | cy=659 (61% down) | ✅ accepted
  offset=+0 | cy=663 (61% down) | ✅ accepted
  offset=+1 | cy=664 (62% down) | ✅ accepted
  offset=+2 | cy=665 (62% down) | ✅ accepted
  offset=+3 | cy=662 (61% down) | ✅ accepted
  offset=+4 | cy=657 (61% down) | ✅ accepted
  offset=+5 | cy=652 (60% down) | ✅ accepted
  offset=+6 | cy=652 (60% down) | ✅ accepted
  offset=+7 | cy=655 (61% down) | ✅ accepted
  offset=+8 | cy=657 (61% down) | ✅ accepted
  offset=+9 | cy=661 (61% down) | ✅ accepted
  offset=+10 | cy=664 (62% down) | ✅ accepted
  offset=+11 | cy=671 (62% down) | ✅ accepted
  offset=+12 | cy=684 (63% down) | ✅ accepted
  offset=+13 | cy=711 (66% down) | ✅ accepted
  offset=+14 | cy=727 (67% down) | ✅ accepted


In [24]:
# ── Section 11: Full Evaluation Across All Chords ────────────────────────────

EXPECTED_NOTES = {
    'videos/c.mov':      ['C4',  'E4',  'G4',  'C5'],
    'videos/d.mov':      ['D4',  'F#4', 'A4',  'D5'],
    'videos/dflat.mov':  ['C#4', 'F4',  'G#4', 'C#5'],
    'videos/e.mov':      ['E4',  'G#4', 'B4',  'E5'],
    'videos/eflat.mov':  ['D#4', 'G4',  'A#4', 'D#5'],
    'videos/f.mov':      ['F4',  'A4',  'C5',  'F5'],
    'videos/gflat.mov':  ['F#4', 'A#4', 'C#5', 'F#5'],
    'videos/g.mov':      ['G4',  'B4',  'D5',  'G5'],
    'videos/glower.mov': ['G3',  'B3',  'D4',  'G4'],
    'videos/a.mov':      ['A3',  'C#4', 'E4',  'A4'],
    'videos/aflat.mov':  ['G#3', 'C4',  'D#4', 'G#4'],
    'videos/b.mov':      ['B3',  'D#4', 'F#4', 'B4'],
    'videos/bflat.mov':  ['A#3', 'D4',  'F4',  'A#4'],

}

results = {}

print('=' * 60)
print('MarimbaVision — Evaluation Results')
print('=' * 60)

for video, expected in EXPECTED_NOTES.items():
    if not os.path.exists(video):
        print(f'\n[SKIP] {video} not found')
        continue

    print(f'\n── {video} ──')
    print(f'   Expected: {expected}')

    strikes = process_video(
        video_path  = video,
        output_path = f'notes_{os.path.basename(video).replace(".mov","")}.json',
        debug       = False
    )

    detected = [s.note for s in strikes]
    methods  = [s.identified_by for s in strikes]
    print(f'   Detected: {detected}')

    n_expected    = len(expected)
    n_detected    = len(detected)
    correct       = 0
    correct_color = 0
    correct_aruco = 0
    correct_pos   = 0
    color_total   = 0
    aruco_total   = 0
    pos_total     = 0

    for exp, det, method in zip(expected, detected, methods):
        is_correct = (exp == det)
        if is_correct:
            correct += 1
        if method == 'color':
            color_total += 1
            if is_correct: correct_color += 1
        elif method == 'aruco_sharp':
            aruco_total += 1
            if is_correct: correct_aruco += 1
        else:
            pos_total += 1
            if is_correct: correct_pos += 1

    accuracy = correct / n_expected * 100 if n_expected > 0 else 0
    print(f'   Count:    {n_detected}/{n_expected} {"✅" if n_detected==n_expected else "❌"}')
    print(f'   Accuracy: {correct}/{n_expected} = {accuracy:.0f}%')
    if color_total > 0:
        print(f'   Color:    {correct_color}/{color_total} = {correct_color/color_total*100:.0f}%')
    if aruco_total > 0:
        print(f'   ArUco:    {correct_aruco}/{aruco_total} = {correct_aruco/aruco_total*100:.0f}%')
    if pos_total > 0:
        print(f'   Position: {correct_pos}/{pos_total} = {correct_pos/pos_total*100:.0f}%')

    results[video] = {
        'expected': expected, 'detected': detected,
        'correct': correct, 'n_expected': n_expected,
        'accuracy': accuracy,
        'color_correct': correct_color, 'color_total': color_total,
        'aruco_correct': correct_aruco, 'aruco_total': aruco_total,
        'pos_correct': correct_pos, 'pos_total': pos_total,
        'methods': methods,
    }

print('\n' + '=' * 60)
print('OVERALL SUMMARY')
print('=' * 60)

total_correct = sum(r['correct']       for r in results.values())
total_notes   = sum(r['n_expected']    for r in results.values())
total_color_c = sum(r['color_correct'] for r in results.values())
total_color_t = sum(r['color_total']   for r in results.values())
total_aruco_c = sum(r['aruco_correct'] for r in results.values())
total_aruco_t = sum(r['aruco_total']   for r in results.values())
total_pos_c   = sum(r['pos_correct']   for r in results.values())
total_pos_t   = sum(r['pos_total']     for r in results.values())

if total_notes > 0:
    print(f'Overall accuracy:      {total_correct}/{total_notes} = {total_correct/total_notes*100:.1f}%')
if total_color_t > 0:
    print(f'Color detection:       {total_color_c}/{total_color_t} = {total_color_c/total_color_t*100:.1f}%')
if total_aruco_t > 0:
    print(f'ArUco sharp detection: {total_aruco_c}/{total_aruco_t} = {total_aruco_c/total_aruco_t*100:.1f}%')
if total_pos_t > 0:
    print(f'Position fallback:     {total_pos_c}/{total_pos_t} = {total_pos_c/total_pos_t*100:.1f}%')

natural_total   = total_color_t
natural_correct = total_color_c
sharp_total     = total_aruco_t + total_pos_t
sharp_correct   = total_aruco_c + total_pos_c

if natural_total > 0:
    print(f'Natural key accuracy:  {natural_correct}/{natural_total} = {natural_correct/natural_total*100:.1f}%')
if sharp_total > 0:
    print(f'Sharp key accuracy:    {sharp_correct}/{sharp_total} = {sharp_correct/sharp_total*100:.1f}%')

print('\nPer-video breakdown:')
print(f'  {"Video":<15} {"Expected":<25} {"Detected":<25} {"Acc":>5}')
print(f'  {"-"*72}')
for video, r in results.items():
    name = video.replace('.mov','')
    exp  = ' '.join(r['expected'])
    det  = ' '.join(r['detected']) if r['detected'] else 'none'
    acc  = f"{r['accuracy']:.0f}%"
    print(f'  {name:<15} {exp:<25} {det:<25} {acc:>5}')

with open('evaluation_summary.json', 'w') as f:
    json.dump({'overall': total_correct/total_notes*100 if total_notes else 0,
               'per_video': {k: {**v, 'methods': v['methods']} 
                             for k,v in results.items()}}, f, indent=2, default=str)
print('\nSaved evaluation_summary.json')

MarimbaVision — Evaluation Results

── videos/c.mov ──
   Expected: ['C4', 'E4', 'G4', 'C5']
[INFO] c.mov | 60.0fps | 6.0s

[STEP 1] Audio onset detection...
[AUDIO] Extracted to temp_audio.wav
[AUDIO] Raw onsets: 4 → filtered: 4
  onset 1: 2.682s
  onset 2: 3.181s
  onset 3: 3.657s
  onset 4: 4.145s
[AUDIO] Found 4 strikes

[STEP 2] Key detection...
[INFO] ArUco markers: [2, 3, 0, 1]
[INFO] 25 keys mapped

[STEP 3] Identifying notes at each onset...
  🎵 Strike 1 @ 2.682s → C4 [color]
  🎵 Strike 2 @ 3.181s → E4 [color]
  [SHARP] Detected: ['A#3', 'G#3', 'D#4', 'C#4', 'F#4', 'D#5', 'A#4', 'C#5', 'F#5', 'G#4']
  [SHARP] Mallet X=916, closest=G#4 (dist=30px)
  🎵 Strike 3 @ 3.657s → G#4 [aruco_sharp]
  🎵 Strike 4 @ 4.145s → C5 [color]

[DONE] 4 strikes → notes_c.json
   Detected: ['C4', 'E4', 'G#4', 'C5']
   Count:    4/4 ✅
   Accuracy: 3/4 = 75%
   Color:    3/3 = 100%
   ArUco:    0/1 = 0%

── videos/d.mov ──
   Expected: ['D4', 'F#4', 'A4', 'D5']
[INFO] d.mov | 60.0fps | 5.6s

[STEP 1] 

In [25]:
"""
MarimbaVision Demo Visuals
Generates:
1. key_detection_demo.mp4 - annotated video showing key detection
2. onset_detection.png - waveform with onset markers
"""

import cv2
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import subprocess
import os

# ── CONFIG — update these paths ──────────────────────────────────────────────
VIDEO_PATH = 'videos/aflat.mov'   # use your best video
OUTPUT_KEY_VIDEO = 'key_detection_demo.mp4'
OUTPUT_ONSET_PNG = 'onset_detection.png'
ONSET_DELTA = 0.3
ONSET_MIN_GAP = 0.3

# ── 1. ONSET DETECTION VISUALIZATION ─────────────────────────────────────────
def make_onset_visualization(video_path, output_path):
    print('[VIZ] Generating onset detection visualization...')
    
    # Extract audio
    subprocess.run(['ffmpeg', '-y', '-i', video_path,
                    '-vn', '-acodec', 'pcm_s16le',
                    '-ar', '44100', '-ac', '1', 'temp_viz_audio.wav'],
                   capture_output=True)
    
    y, sr = librosa.load('temp_viz_audio.wav', sr=None, mono=True)
    
    # Compute onset strength
    onset_env    = librosa.onset.onset_strength(y=y, sr=sr)
    onset_frames = librosa.onset.onset_detect(
        y=y, sr=sr, onset_envelope=onset_env,
        delta=ONSET_DELTA, units='frames')
    onset_times  = librosa.frames_to_time(onset_frames, sr=sr)
    
    # Filter by min gap
    filtered = []
    last_t   = -999
    for t in onset_times:
        if t - last_t >= ONSET_MIN_GAP:
            filtered.append(float(t))
            last_t = t
    
    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    fig.patch.set_facecolor('#1a1a2e')
    
    times = np.linspace(0, len(y)/sr, len(y))
    
    # Panel 1: Raw waveform
    ax1 = axes[0]
    ax1.set_facecolor('#16213e')
    ax1.plot(times, y, color='#00d4ff', linewidth=0.5, alpha=0.8)
    for t in filtered:
        ax1.axvline(x=t, color='#ff6b6b', linewidth=2, alpha=0.9)
    ax1.set_ylabel('Amplitude', color='white', fontsize=11)
    ax1.set_title('Raw Audio Waveform', color='white', fontsize=12, pad=8)
    ax1.tick_params(colors='gray')
    ax1.spines[:].set_color('#333366')
    
    # Panel 2: Onset strength envelope
    ax2 = axes[1]
    ax2.set_facecolor('#16213e')
    onset_times_full = librosa.frames_to_time(
        np.arange(len(onset_env)), sr=sr)
    ax2.plot(onset_times_full, onset_env,
             color='#ffd700', linewidth=1.5, label='Onset strength')
    ax2.axhline(y=onset_env.max() * ONSET_DELTA,
                color='#ff6b6b', linewidth=1.5,
                linestyle='--', alpha=0.7, label=f'Threshold (δ={ONSET_DELTA})')
    for t in filtered:
        ax2.axvline(x=t, color='#ff6b6b', linewidth=2, alpha=0.9)
    ax2.set_ylabel('Onset Strength', color='white', fontsize=11)
    ax2.set_title('Librosa Onset Strength Envelope', color='white',
                  fontsize=12, pad=8)
    ax2.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
    ax2.tick_params(colors='gray')
    ax2.spines[:].set_color('#333366')
    
    # Panel 3: Spectrogram
    ax3 = axes[2]
    ax3.set_facecolor('#16213e')
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz',
                              ax=ax3, cmap='magma')
    for i, t in enumerate(filtered):
        ax3.axvline(x=t, color='#ff6b6b', linewidth=2, alpha=0.9)
        ax3.text(t + 0.02, sr//4,
                 f'Strike {i+1}', color='white',
                 fontsize=8, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.2',
                           facecolor='#ff6b6b', alpha=0.7))
    ax3.set_ylabel('Frequency (Hz)', color='white', fontsize=11)
    ax3.set_xlabel('Time (s)', color='white', fontsize=11)
    ax3.set_title('Spectrogram with Detected Strikes', color='white',
                  fontsize=12, pad=8)
    ax3.tick_params(colors='gray')
    ax3.spines[:].set_color('#333366')
    
    # Legend
    strike_patch = mpatches.Patch(color='#ff6b6b',
                                   label=f'{len(filtered)} strikes detected')
    fig.legend(handles=[strike_patch], loc='upper right',
               facecolor='#1a1a2e', labelcolor='white', fontsize=10)
    
    fig.suptitle('MarimbaVision — Audio Onset Detection',
                 color='white', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight',
                facecolor='#1a1a2e')
    plt.close()
    
    os.remove('temp_viz_audio.wav')
    print(f'[VIZ] Saved {output_path}')
    return filtered


# ── 2. KEY DETECTION DEMO VIDEO ───────────────────────────────────────────────
def make_key_detection_video(video_path, strike_times, output_path):
    print('[VIZ] Generating key detection demo video...')
    
    cap = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fw    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(output_path,
                             cv2.VideoWriter_fourcc(*'mp4v'),
                             fps, (fw, fh))

    # ArUco setup
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    params     = cv2.aruco.DetectorParameters()
    params.minMarkerPerimeterRate      = 0.02
    params.polygonalApproxAccuracyRate = 0.08
    detector   = cv2.aruco.ArucoDetector(aruco_dict, params)

    SHARP_ID_TO_NOTE = {
        5:'G#3', 7:'A#3', 10:'C#4', 12:'D#4', 15:'F#4',
        17:'G#4', 19:'A#4', 22:'C#5', 24:'D#5', 27:'F#5'
    }

    # Mallet HSV
    MALLET_LOWER = np.array([125, 50, 100])
    MALLET_UPPER = np.array([155, 255, 255])

    # Flash strike info
    strike_flash = {}
    for t in strike_times:
        fn = int(t * fps)
        for offset in range(-3, 10):
            strike_flash[fn + offset] = t

    frame_no  = 0
    keys_cache = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        vis = frame.copy()
        t_now = frame_no / fps

        # ── Detect ArUco markers ───────────────────────────────────────────
        corners, ids, _ = detector.detectMarkers(frame)
        marker_centers  = {}
        if ids is not None:
            for i, corner in enumerate(corners):
                mid = int(ids[i][0])
                cx  = int(corner[0].mean(axis=0)[0])
                cy  = int(corner[0].mean(axis=0)[1])
                marker_centers[mid] = (cx, cy)

                # Draw corner markers
                if mid <= 3:
                    cv2.drawMarker(vis, (cx, cy), (0, 0, 255),
                                   cv2.MARKER_STAR, 20, 2)
                    cv2.putText(vis, f'ID={mid}', (cx+8, cy-8),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.45, (0, 0, 255), 1)
                # Draw sharp key markers
                elif mid in SHARP_ID_TO_NOTE:
                    cv2.circle(vis, (cx, cy), 5, (255, 100, 0), -1)
                    cv2.putText(vis, SHARP_ID_TO_NOTE[mid],
                                (cx-15, cy-8),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.35, (255, 180, 0), 1)

        # ── Draw vanishing point lines if all 4 corners present ───────────
        if all(i in marker_centers for i in range(4)):
            for a, b in [(0, 1), (2, 3)]:
                cv2.line(vis, marker_centers[a], marker_centers[b],
                         (0, 200, 255), 1, cv2.LINE_AA)
            for a, b in [(0, 2), (1, 3)]:
                cv2.line(vis, marker_centers[a], marker_centers[b],
                         (0, 200, 100), 1, cv2.LINE_AA)

        # ── Track mallet ──────────────────────────────────────────────────
        hsv    = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask   = cv2.inRange(hsv, MALLET_LOWER, MALLET_UPPER)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
        mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        conts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                     cv2.CHAIN_APPROX_SIMPLE)
        best_cnt, best_area = None, 0
        for cnt in conts:
            area = cv2.contourArea(cnt)
            if not (500 < area < 8000):
                continue
            perim = cv2.arcLength(cnt, True)
            if perim == 0:
                continue
            if 4*np.pi*area/(perim**2) < 0.6:
                continue
            if area > best_area:
                best_area, best_cnt = area, cnt

        mallet_pos = None
        if best_cnt is not None:
            (cx, cy), r = cv2.minEnclosingCircle(best_cnt)
            cx, cy, r   = int(cx), int(cy), int(r)
            mallet_pos  = (cx, cy)
            cv2.circle(vis, (cx, cy), r, (0, 0, 255), 2)
            cv2.circle(vis, (cx, cy), 3, (255, 255, 255), -1)

        # ── Strike flash overlay ──────────────────────────────────────────
        if frame_no in strike_flash:
            overlay = vis.copy()
            cv2.rectangle(overlay, (0, 0), (fw, fh), (0, 0, 255), -1)
            cv2.addWeighted(overlay, 0.15, vis, 0.85, 0, vis)
            cv2.putText(vis, '♪ STRIKE DETECTED',
                        (fw//2 - 150, 60),
                        cv2.FONT_HERSHEY_DUPLEX,
                        1.2, (0, 255, 255), 2)

        # ── HUD ───────────────────────────────────────────────────────────
        # Time bar
        bar_w = int((t_now / (total/fps)) * fw)
        cv2.rectangle(vis, (0, fh-8), (fw, fh),  (40, 40, 40), -1)
        cv2.rectangle(vis, (0, fh-8), (bar_w, fh), (0, 200, 255), -1)

        # Strike markers on timeline
        for st in strike_times:
            sx = int((st / (total/fps)) * fw)
            cv2.rectangle(vis, (sx-2, fh-8), (sx+2, fh),
                          (0, 0, 255), -1)

        # Info panel
        cv2.rectangle(vis, (0, 0), (320, 95), (0, 0, 0), -1)
        cv2.rectangle(vis, (0, 0), (320, 95), (0, 200, 255), 1)
        cv2.putText(vis, 'MarimbaVision',
                    (10, 22), cv2.FONT_HERSHEY_DUPLEX,
                    0.65, (0, 200, 255), 1)
        cv2.putText(vis, f'Time: {t_now:.2f}s',
                    (10, 45), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (200, 200, 200), 1)
        n_markers = len([k for k in marker_centers if k <= 3])
        cv2.putText(vis,
                    f'Corner markers: {n_markers}/4',
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 255, 0) if n_markers == 4 else (0, 100, 255), 1)
        sharp_visible = len([k for k in marker_centers
                              if k in SHARP_ID_TO_NOTE])
        cv2.putText(vis,
                    f'Sharp markers: {sharp_visible}/10',
                    (10, 85), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (255, 180, 0), 1)

        writer.write(vis)
        frame_no += 1

        if frame_no % 60 == 0:
            print(f'  {frame_no/total*100:.0f}%')

    cap.release()
    writer.release()
    print(f'[VIZ] Saved {output_path}')


# ── MAIN ──────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    strike_times = make_onset_visualization(VIDEO_PATH, OUTPUT_ONSET_PNG)
    make_key_detection_video(VIDEO_PATH, strike_times, OUTPUT_KEY_VIDEO)
    print('\n✅ Both files saved!')
    print(f'  {OUTPUT_ONSET_PNG}')
    print(f'  {OUTPUT_KEY_VIDEO}')

[VIZ] Generating onset detection visualization...
[VIZ] Saved onset_detection.png
[VIZ] Generating key detection demo video...
  18%
  35%
  53%
  70%
  88%
[VIZ] Saved key_detection_demo.mp4

✅ Both files saved!
  onset_detection.png
  key_detection_demo.mp4


In [26]:
# ── Demo Visualization Cell ───────────────────────────────────────────────────
import cv2
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import subprocess
import os
import json

# ── CONFIG ────────────────────────────────────────────────────────────────────
DEMO_VIDEO      = 'videos/aflat.mov'
NOTES_JSON      = 'notes_aflat.json'
OUTPUT_VIDEO    = 'key_detection_demo.mp4'
OUTPUT_ONSET    = 'onset_detection.png'
ONSET_DELTA     = 0.3
ONSET_MIN_GAP   = 0.3

SHARP_ID_TO_NOTE = {
    5:'G#3', 7:'A#3', 10:'C#4', 12:'D#4', 15:'F#4',
    17:'G#4', 19:'A#4', 22:'C#5', 24:'D#5', 27:'F#5'
}
MALLET_LOWER = np.array([125, 50,  100])
MALLET_UPPER = np.array([155, 255, 255])

# ── Load detected notes from JSON ─────────────────────────────────────────────
with open(NOTES_JSON) as f:
    data = json.load(f)
strike_notes = {round(s['timestamp'], 3): s['note'] for s in data['strikes']}
strike_times = sorted(strike_notes.keys())
print(f'Loaded {len(strike_notes)} strikes: {strike_notes}')

# ── PART 1: Onset Detection PNG ───────────────────────────────────────────────
print('\n[1/2] Generating onset visualization...')
subprocess.run(['ffmpeg', '-y', '-i', DEMO_VIDEO,
                '-vn', '-acodec', 'pcm_s16le',
                '-ar', '44100', '-ac', '1', 'temp_demo_audio.wav'],
               capture_output=True)

y, sr = librosa.load('temp_demo_audio.wav', sr=None, mono=True)
onset_env    = librosa.onset.onset_strength(y=y, sr=sr)
onset_frames = librosa.onset.onset_detect(y=y, sr=sr,
                    onset_envelope=onset_env, delta=ONSET_DELTA, units='frames')
onset_times_all = librosa.frames_to_time(onset_frames, sr=sr)
filtered = []
last_t = -999
for t in onset_times_all:
    if t - last_t >= ONSET_MIN_GAP:
        filtered.append(float(t))
        last_t = t

fig, axes = plt.subplots(3, 1, figsize=(14, 8))
fig.patch.set_facecolor('#1a1a2e')
times = np.linspace(0, len(y)/sr, len(y))

# Waveform
ax1 = axes[0]
ax1.set_facecolor('#16213e')
ax1.plot(times, y, color='#00d4ff', linewidth=0.5, alpha=0.8)
for i, t in enumerate(filtered):
    note = strike_notes.get(round(t, 3), '')
    ax1.axvline(x=t, color='#ff6b6b', linewidth=2, alpha=0.9)
    ax1.text(t+0.02, ax1.get_ylim()[1]*0.7 if ax1.get_ylim()[1] != 0 else 0.3,
             note, color='#ff6b6b', fontsize=9, fontweight='bold')
ax1.set_ylabel('Amplitude', color='white', fontsize=11)
ax1.set_title('Raw Audio Waveform', color='white', fontsize=12)
ax1.tick_params(colors='gray')
ax1.spines[:].set_color('#333366')

# Onset envelope
ax2 = axes[1]
ax2.set_facecolor('#16213e')
onset_t = librosa.frames_to_time(np.arange(len(onset_env)), sr=sr)
ax2.plot(onset_t, onset_env, color='#ffd700', linewidth=1.5,
         label='Onset strength')
ax2.axhline(y=onset_env.max()*ONSET_DELTA, color='#ff6b6b',
            linewidth=1.5, linestyle='--', alpha=0.7,
            label=f'Threshold (δ={ONSET_DELTA})')
for t in filtered:
    ax2.axvline(x=t, color='#ff6b6b', linewidth=2, alpha=0.9)
ax2.set_ylabel('Onset Strength', color='white', fontsize=11)
ax2.set_title('Librosa Onset Strength Envelope', color='white', fontsize=12)
ax2.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
ax2.tick_params(colors='gray')
ax2.spines[:].set_color('#333366')

# Spectrogram
ax3 = axes[2]
ax3.set_facecolor('#16213e')
D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz',
                          ax=ax3, cmap='magma')
for i, t in enumerate(filtered):
    note = strike_notes.get(round(t, 3), f'Strike {i+1}')
    ax3.axvline(x=t, color='#ff6b6b', linewidth=2, alpha=0.9)
    ax3.text(t+0.02, 8000, note, color='white', fontsize=9,
             fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2',
                       facecolor='#ff6b6b', alpha=0.8))
ax3.set_ylabel('Frequency (Hz)', color='white', fontsize=11)
ax3.set_xlabel('Time (s)', color='white', fontsize=11)
ax3.set_title('Spectrogram with Detected Strikes', color='white', fontsize=12)
ax3.tick_params(colors='gray')
ax3.spines[:].set_color('#333366')

fig.suptitle('MarimbaVision — Audio Onset Detection',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_ONSET, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.close()
os.remove('temp_demo_audio.wav')
print(f'  Saved {OUTPUT_ONSET}')

# ── PART 2: Key Detection Demo Video ─────────────────────────────────────────
print('\n[2/2] Generating key detection demo video...')

cap   = cv2.VideoCapture(DEMO_VIDEO)
fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fw    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fh    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(OUTPUT_VIDEO,
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          fps, (fw, fh))

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
params     = cv2.aruco.DetectorParameters()
params.minMarkerPerimeterRate      = 0.02
params.polygonalApproxAccuracyRate = 0.08
detector   = cv2.aruco.ArucoDetector(aruco_dict, params)

# Build strike flash lookup — flash for 15 frames around each strike
strike_flash = {}
for t, note in strike_notes.items():
    fn = int(t * fps)
    for offset in range(-2, 15):
        strike_flash[fn + offset] = note

frame_no = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    vis  = frame.copy()
    t_now = frame_no / fps

    # Detect markers
    corners, ids, _ = detector.detectMarkers(frame)
    marker_centers  = {}
    if ids is not None:
        for i, corner in enumerate(corners):
            mid = int(ids[i][0])
            cx  = int(corner[0].mean(axis=0)[0])
            cy  = int(corner[0].mean(axis=0)[1])
            marker_centers[mid] = (cx, cy)
            if mid <= 3:
                cv2.drawMarker(vis, (cx,cy), (0,0,255),
                               cv2.MARKER_STAR, 20, 2)
                cv2.putText(vis, f'ID={mid}', (cx+8,cy-8),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.45, (0,0,255), 1)
            elif mid in SHARP_ID_TO_NOTE:
                cv2.circle(vis, (cx,cy), 5, (255,130,0), -1)
                cv2.putText(vis, SHARP_ID_TO_NOTE[mid],
                            (cx-15,cy-8),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.35, (255,180,0), 1)

    # Vanishing point lines
    if all(i in marker_centers for i in range(4)):
        for a, b in [(0,1),(2,3)]:
            cv2.line(vis, marker_centers[a], marker_centers[b],
                     (0,200,255), 1, cv2.LINE_AA)
        for a, b in [(0,2),(1,3)]:
            cv2.line(vis, marker_centers[a], marker_centers[b],
                     (0,200,100), 1, cv2.LINE_AA)

    # Track mallet
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, MALLET_LOWER, MALLET_UPPER)
    k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    conts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                 cv2.CHAIN_APPROX_SIMPLE)
    best_cnt, best_area = None, 0
    for cnt in conts:
        area = cv2.contourArea(cnt)
        if not (500 < area < 8000): continue
        perim = cv2.arcLength(cnt, True)
        if perim == 0: continue
        if 4*np.pi*area/(perim**2) < 0.6: continue
        if area > best_area:
            best_area, best_cnt = area, cnt

    mallet_pos = None
    if best_cnt is not None:
        (cx,cy), r = cv2.minEnclosingCircle(best_cnt)
        cx,cy,r    = int(cx), int(cy), int(r)
        mallet_pos = (cx, cy)
        cv2.circle(vis, (cx,cy), r, (0,0,255), 2)
        cv2.circle(vis, (cx,cy), 3, (255,255,255), -1)

    # Strike flash with note label
    if frame_no in strike_flash:
        note = strike_flash[frame_no]
        overlay = vis.copy()
        cv2.rectangle(overlay, (0,0), (fw,fh), (0,0,200), -1)
        cv2.addWeighted(overlay, 0.15, vis, 0.85, 0, vis)
        cv2.putText(vis, 'STRIKE DETECTED',
                    (fw//2-160, 55),
                    cv2.FONT_HERSHEY_DUPLEX,
                    1.2, (0,255,255), 2)
        cv2.putText(vis, f'Note: {note}',
                    (fw//2-90, 105),
                    cv2.FONT_HERSHEY_DUPLEX,
                    1.8, (0,255,100), 3)
        if mallet_pos:
            cx, cy = mallet_pos
            cv2.circle(vis, (cx,cy), 55, (0,255,100), 3)
            cv2.putText(vis, note, (cx+60, cy+10),
                        cv2.FONT_HERSHEY_DUPLEX,
                        1.2, (0,255,100), 2)

    # Timeline bar
    bar_w = int((t_now / (total/fps)) * fw)
    cv2.rectangle(vis, (0,fh-8), (fw,fh), (40,40,40), -1)
    cv2.rectangle(vis, (0,fh-8), (bar_w,fh), (0,200,255), -1)
    for st in strike_times:
        sx = int((st / (total/fps)) * fw)
        cv2.rectangle(vis, (sx-2,fh-8), (sx+2,fh), (0,0,255), -1)

    # HUD
    cv2.rectangle(vis, (0,0), (320,95), (0,0,0), -1)
    cv2.rectangle(vis, (0,0), (320,95), (0,200,255), 1)
    cv2.putText(vis, 'MarimbaVision',
                (10,22), cv2.FONT_HERSHEY_DUPLEX,
                0.65, (0,200,255), 1)
    cv2.putText(vis, f'Time: {t_now:.2f}s',
                (10,45), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (200,200,200), 1)
    n_corners = len([k for k in marker_centers if k <= 3])
    n_sharps  = len([k for k in marker_centers if k in SHARP_ID_TO_NOTE])
    cv2.putText(vis, f'Corner markers: {n_corners}/4',
                (10,65), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                (0,255,0) if n_corners==4 else (0,100,255), 1)
    cv2.putText(vis, f'Sharp markers: {n_sharps}/10',
                (10,85), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (255,180,0), 1)

    writer.write(vis)
    frame_no += 1
    if frame_no % 60 == 0:
        print(f'  {frame_no/total*100:.0f}%')

cap.release()
writer.release()
print(f'  Saved {OUTPUT_VIDEO}')
print('\n✅ Done! Files saved:')
print(f'  {OUTPUT_ONSET}')
print(f'  {OUTPUT_VIDEO}')

Loaded 4 strikes: {2.276: 'G#3', 2.728: 'C4', 3.146: 'D#4', 3.553: 'G#4'}

[1/2] Generating onset visualization...
  Saved onset_detection.png

[2/2] Generating key detection demo video...
  18%
  35%
  53%
  70%
  88%
  Saved key_detection_demo.mp4

✅ Done! Files saved:
  onset_detection.png
  key_detection_demo.mp4
